In [1]:
"""
exp13 Step 02: Parameter Neighbourhood Of E1M1 (PROTOCOL.md §4, step 02; DESCRIPTIVE: nothing is selected from it)

36 variants around the frozen E1M1: fade ratio {0.70, 0.75, 0.80, 0.85, 0.90, 0.95} x moving-average length {150, 200,
250} sessions (exit below it and original re-entry above it) x fresh-signal rule {on, off}; E1M1 = 0.85 / 200 / on. Every
variant runs as one continuous path over DEV (2015-04-17 -> 2026-04-15, exp12's candidate span; SPY and VIX cut at
2026-04-15) and over STRESS (first session with the 200-session average and vix_level_prev -> 2014-12-31; cut at
2015-03-18): 72 paths. Reproduction asserts first (frozen E1M1 +314.80% with 35 exits, E1 +108.03% with 35 exits,
buy-and-hold +235.39%), then the variant at 0.85 / 200 / on must equal the frozen E1M1 exactly.

Reported: the two grids of total return minus buy-and-hold, the share of the 36 variants beating buy-and-hold on DEV, on
STRESS and on both, and E1M1's rank on DEV. It describes the shape of the surface (a plateau or a spike); no variant is
picked or recommended. RUN_MODE: "validation_only" only (the untouched window is never read).
Outputs: store04_experiments/exp13_e1m1_robustness/step02_neighbourhood_data/. Trials: 72 (stage "robustness").
"""
None

In [2]:
# IMPORT GENERAL PACKAGES
import pandas as pd
import numpy as np
import time
# IMPORT THE SHARED CONFIGURATION, THE EXPERIMENT CONFIGURATION AND THE PATHS
from so import config, paths
from experiments.exp13_e1m1_robustness import config as exp_config
# IMPORT LOCAL FILE MANAGEMENT FUNCTIONS
from so.core.local_file_management import write_csv_file_to_path
# IMPORT EXIT AND RE-ENTRY SIMULATION FUNCTIONS
from so.core.reentry_simulation import get_buy_and_hold_result_dict
# IMPORT THE EXPERIMENT FUNCTIONS
from experiments.exp13_e1m1_robustness.robustness import get_span_data_dict, get_dev_period_pdf, check_last_date_dict, get_stress_start_date, get_year_period_pdf, \
    get_dev_reproduction_dict, get_variant_rule_tuple, replay_rule_dict, get_path_metric_dict, check_step_trial_dict
# IMPORT TRIAL LOG FUNCTIONS
from so.core.trial_log import get_config_hash_str, log_trial_dict, read_trial_log_pdf, get_trial_count_pdf, print_previous_trial_warning

In [3]:
# DEFINE THE RUN MODE: ONLY "validation_only" EXISTS FOR THIS EXPERIMENT (NO TEST WINDOW IS EVALUATED)
RUN_MODE_STR = "validation_only"
assert RUN_MODE_STR == "validation_only"
# CHECK THE REAL DATA ONLY OUTSIDE THE SYNTHETIC SMOKE RUN (scripts/smoke_notebook.py)
CHECK_REAL_DATA_BOOL = __name__ != "__smoke__"
# DISPLAY THE EXPERIMENT, THE PROTOCOL VERSION AND THE CONFIGURATION HASH (SHARED + EXPERIMENT CONFIG)
print(f"Run Mode:\t{RUN_MODE_STR}\tExperiment:\t{exp_config.EXPERIMENT_NAME}\tProtocol:\t{exp_config.PROTOCOL_VERSION}\tConfig Hash:\t{get_config_hash_str(exp_config)}")
# DEFINE THE OUTPUT FOLDER OF THIS STEP
output_path_str = paths.get_experiment_data_path_str(exp_config.EXPERIMENT_NAME, "step02_neighbourhood_data")
print(f"Output folder:\t{output_path_str}")
# CALL FUNCTION TO WARN ABOUT EARLIER ENTRIES OF THIS STAGE (A RERUN ADDS TRIALS)
previous_trial_count = print_previous_trial_warning(exp_config, "robustness")

Run Mode:	validation_only	Experiment:	exp13_e1m1_robustness	Protocol:	1.0	Config Hash:	c2e293d595
Output folder:	C:/Users/nico/Desktop/stock_overflow_data/store04_experiments/exp13_e1m1_robustness/step02_neighbourhood_data/
ℹ️ No earlier 'robustness' entry of exp13_e1m1_robustness in the trial log (config hash c2e293d595).


In [4]:
# CALL FUNCTION TO READ exp12's 44 REPLAY PERIODS (DATES ONLY) AND LOAD SPY AND THE VIX UP TO THEIR END (2026-04-15)
dev_period_pdf = get_dev_period_pdf(CHECK_REAL_DATA_BOOL)
dev_cutoff_date_str = str(dev_period_pdf["period_end"].iloc[-1])
dev_data_dict = get_span_data_dict(dev_cutoff_date_str)
print(f"DEV:\t{dev_period_pdf['period_start'].iloc[0]} -> {dev_cutoff_date_str}\t({len(dev_period_pdf)} periods)\tSPY sessions loaded {len(dev_data_dict['daily_pdf']):,} from {dev_data_dict['first_spy_date']}")
dev_last_date_dict = check_last_date_dict(dev_data_dict["last_date_dict"], dev_cutoff_date_str, CHECK_REAL_DATA_BOOL)
# CALL FUNCTION TO LOAD SPY AND THE VIX UP TO THE STRESS CUTOFF (2015-03-18)
stress_data_dict = get_span_data_dict(exp_config.STRESS_DATA_CUTOFF_DATE_STR)
stress_start_date = get_stress_start_date(stress_data_dict["daily_pdf"], stress_data_dict["vix_feature_pdf"])
print(f"STRESS:\t{stress_start_date} -> {exp_config.STRESS_END_DATE_STR}\tSPY sessions loaded {len(stress_data_dict['daily_pdf']):,} from {stress_data_dict['first_spy_date']}")
stress_last_date_dict = check_last_date_dict(stress_data_dict["last_date_dict"], exp_config.STRESS_DATA_CUTOFF_DATE_STR, CHECK_REAL_DATA_BOOL)

DEV:	2015-04-17 -> 2026-04-15	(44 periods)	SPY sessions loaded 5,353 from 2005-01-03
Last date loaded:	spy	2026-04-15	(cutoff 2026-04-15)
Last date loaded:	vix_daily	2026-04-15	(cutoff 2026-04-15)
Last date loaded:	vix3m_daily	2026-04-15	(cutoff 2026-04-15)


STRESS:	2005-10-18 -> 2014-12-31	SPY sessions loaded 2,568 from 2005-01-03
Last date loaded:	spy	2015-03-18	(cutoff 2015-03-18)
Last date loaded:	vix_daily	2015-03-18	(cutoff 2015-03-18)
Last date loaded:	vix3m_daily	2015-03-18	(cutoff 2015-03-18)


In [5]:
"""
Reproduction (before any exp13 number): frozen E1M1, unmodified E1 and buy-and-hold over DEV (exp12's code).
"""
# CALL FUNCTION TO REPLAY AND ASSERT THE REPRODUCTION PATHS
dev_reproduction_dict = get_dev_reproduction_dict(dev_data_dict, dev_period_pdf, CHECK_REAL_DATA_BOOL)
print(f"Reproduction:\t{dev_reproduction_dict['reproduction_dict']}")
print("Reproduction asserts:\tPASSED") if CHECK_REAL_DATA_BOOL else None

Reproduction:	{'dev_start': '2015-04-17', 'dev_end': '2026-04-15', 'e1m1_total_pct': 314.8, 'e1m1_exit_count': 35, 'e1_total_pct': 108.03, 'e1_exit_count': 35, 'buy_hold_pct': 235.39}
Reproduction asserts:	PASSED


In [6]:
"""
The 36 variants over DEV and over STRESS (72 continuous paths), each against buy-and-hold over the same sessions.
"""
# DEFINE THE GRID AND THE TWO SPANS
variant_dict_list = [{"fade_ratio": fade_ratio, "ma_length": ma_length, "fresh_exit": fresh_exit_bool}
                     for ma_length in exp_config.MA_LENGTH_LIST for fresh_exit_bool in exp_config.FRESH_EXIT_BOOL_LIST for fade_ratio in exp_config.FADE_RATIO_LIST]
span_dict = {"dev": (dev_data_dict, dev_reproduction_dict["date1"], dev_reproduction_dict["date2"], dev_period_pdf),
             "stress": (stress_data_dict, stress_start_date, pd.Timestamp(exp_config.STRESS_END_DATE_STR).date(),
                        get_year_period_pdf(stress_data_dict["ohlcv_array_dict"]["session_date_list"], stress_start_date, exp_config.STRESS_END_DATE_STR))}
# LIST TO HOLD THE ROWS
start_time = time.time()
row_dict_list = []
# ITERATE OVER THE SPANS AND THE VARIANTS
for span_str, (data_dict, date1, date2, period_pdf) in span_dict.items():
    buy_hold_dict = get_buy_and_hold_result_dict(data_dict["ohlcv_array_dict"], date1, date2)
    for variant_dict in variant_dict_list:
        rule_tuple = get_variant_rule_tuple(data_dict["daily_pdf"], data_dict["vix_feature_pdf"], variant_dict["fade_ratio"], variant_dict["ma_length"], variant_dict["fresh_exit"])
        simulation_dict = replay_rule_dict(data_dict["ohlcv_array_dict"], rule_tuple, date1, date2)
        trigger_series = simulation_dict["episode_pdf"]["reentry_trigger"] if len(simulation_dict["episode_pdf"]) and "reentry_trigger" in simulation_dict["episode_pdf"].columns else pd.Series(dtype=str)
        row_dict_list.append({"span": span_str, **variant_dict, "is_e1m1": bool(variant_dict["fade_ratio"] == exp_config.FROZEN_M1_FADE_RATIO and variant_dict["ma_length"] == exp_config.FROZEN_MA_LENGTH
                                                                                and variant_dict["fresh_exit"] == exp_config.FROZEN_FRESH_EXIT_BOOL),
                              **get_path_metric_dict(simulation_dict, buy_hold_dict, period_pdf, data_dict["ohlcv_array_dict"]), "m1_reentry_count": int((trigger_series == "M1").sum())})
grid_pdf = pd.DataFrame(row_dict_list)
print(f"72 paths replayed in {time.time() - start_time:.0f} s")
# ASSERT THAT THE VARIANT AT 0.85 / 200 / ON IS THE FROZEN E1M1 (SAME TOTAL AND EXITS ON DEV)
e1m1_dev_row = grid_pdf[(grid_pdf["span"] == "dev") & grid_pdf["is_e1m1"]].iloc[0]
assert e1m1_dev_row["total_return"] == dev_reproduction_dict["frozen_simulation_dict"]["metric_dict"]["total_return"], "❌ The grid's E1M1 differs from the frozen E1M1"
assert e1m1_dev_row["exit_count"] == dev_reproduction_dict["frozen_simulation_dict"]["metric_dict"]["signal_exit_count"], "❌ The grid's E1M1 differs from the frozen E1M1"
print(f"Grid E1M1 = frozen E1M1:\t{e1m1_dev_row['total_return']:.2%} with {int(e1m1_dev_row['exit_count'])} exits")

72 paths replayed in 67 s
Grid E1M1 = frozen E1M1:	314.80% with 35 exits


In [7]:
"""
The surface: total return minus buy-and-hold (percentage points) per variant, rows = average length and fresh-signal
rule, columns = fade ratio. Descriptive only.
"""
# BUILD THE TWO GRIDS
grid_table_dict = {}
for span_str in span_dict:
    span_grid_pdf = grid_pdf[grid_pdf["span"] == span_str]
    grid_table_dict[span_str] = (100 * span_grid_pdf.pivot_table(index=["ma_length", "fresh_exit"], columns="fade_ratio", values="excess_total_return")).round(2)
    print(f"\n{span_str.upper()}: total return minus buy-and-hold ({100 * span_grid_pdf['buy_hold_total_return'].iloc[0]:.2f}%), percentage points")
    display(grid_table_dict[span_str])
# COUNT THE VARIANTS BEATING BUY-AND-HOLD AND RANK E1M1 ON DEV
beat_pdf = grid_pdf.pivot_table(index=["fade_ratio", "ma_length", "fresh_exit"], columns="span", values="beats_buy_hold", aggfunc="first")
dev_grid_pdf = grid_pdf[grid_pdf["span"] == "dev"].copy()
dev_grid_pdf["dev_rank"] = dev_grid_pdf["total_return"].rank(ascending=False, method="min").astype(int)
neighbourhood_dict = {"variant_count": int(len(beat_pdf)), "dev_beat_count": int(beat_pdf["dev"].sum()), "stress_beat_count": int(beat_pdf["stress"].sum()),
                      "both_beat_count": int((beat_pdf["dev"] & beat_pdf["stress"]).sum()), "e1m1_dev_rank": int(dev_grid_pdf.loc[dev_grid_pdf["is_e1m1"], "dev_rank"].iloc[0]),
                      "e1m1_dev_total_return": float(e1m1_dev_row["total_return"]), "dev_buy_hold_total_return": float(e1m1_dev_row["buy_hold_total_return"]),
                      "stress_buy_hold_total_return": float(grid_pdf.loc[grid_pdf["span"] == "stress", "buy_hold_total_return"].iloc[0]),
                      "dev_median_excess": float(grid_pdf.loc[grid_pdf["span"] == "dev", "excess_total_return"].median()),
                      "stress_median_excess": float(grid_pdf.loc[grid_pdf["span"] == "stress", "excess_total_return"].median())}
print(f"\nVariants beating buy-and-hold:\tDEV {neighbourhood_dict['dev_beat_count']} of 36\tSTRESS {neighbourhood_dict['stress_beat_count']} of 36\t"
      f"both {neighbourhood_dict['both_beat_count']} of 36\tE1M1's rank on DEV: {neighbourhood_dict['e1m1_dev_rank']} of 36")
print(f"Median excess:\tDEV {neighbourhood_dict['dev_median_excess']:.2%}\tSTRESS {neighbourhood_dict['stress_median_excess']:.2%}")
# PREVIEW DATAFRAME
grid_pdf[["span", "fade_ratio", "ma_length", "fresh_exit", "total_return", "buy_hold_total_return", "excess_total_return", "annualized_log_excess", "log_excess_ci_low_6m",
          "log_excess_ci_high_6m", "in_market_pct", "exit_count", "m1_reentry_count", "product_s_over_r", "sharpe_ratio", "max_drawdown"]].round(4)


DEV: total return minus buy-and-hold (235.39%), percentage points


fade_ratio              0.70    0.75    0.80    0.85    0.90    0.95
ma_length fresh_exit                                                
150       False      -162.64 -149.57 -129.72 -113.26 -148.35 -122.04
          True        -57.61  -27.73   -5.97    7.93  -28.83  -39.62
200       False      -137.41 -120.76 -103.49  -96.40 -129.57 -112.18
          True          0.47   19.00   50.27   79.41   15.75   12.83
250       False      -133.29 -119.32  -93.56  -89.58 -115.14  -84.15
          True         -5.73    1.05   21.53   59.77    3.48   -0.37


STRESS: total return minus buy-and-hold (73.09%), percentage points


fade_ratio             0.70   0.75   0.80   0.85   0.90   0.95
ma_length fresh_exit                                          
150       False      -40.15 -27.26  -3.42 -27.22 -46.27 -76.45
          True       -42.41 -39.46 -34.31 -24.86 -18.77 -26.09
200       False      -44.51 -35.98  -3.04 -26.24 -48.57 -70.56
          True       -51.30 -51.69 -22.59 -19.77   1.26 -12.21
250       False      -25.72 -15.45  14.68  -0.37 -22.83 -60.54
          True       -34.01 -36.31 -26.43 -25.50   5.55  -9.20


Variants beating buy-and-hold:	DEV 11 of 36	STRESS 3 of 36	both 2 of 36	E1M1's rank on DEV: 1 of 36
Median excess:	DEV -70.88%	STRESS -26.34%


,span,fade_ratio,ma_length,fresh_exit,total_return,buy_hold_total_return,excess_total_return,annualized_log_excess,log_excess_ci_low_6m,log_excess_ci_high_6m,in_market_pct,exit_count,m1_reentry_count,product_s_over_r,sharpe_ratio,max_drawdown
0,dev,0.70,150,True,1.7778,2.3539,-0.5761,-0.0170,-0.0534,0.0259,0.8618,54,7,0.8299,0.7570,-0.2215
1,dev,0.75,150,True,2.0766,2.3539,-0.2773,-0.0078,-0.0473,0.0415,0.8755,54,8,0.9190,0.8034,-0.2383
2,dev,0.80,150,True,2.2942,2.3539,-0.0597,-0.0016,-0.0428,0.0497,0.8889,54,11,0.9842,0.8078,-0.2077
3,dev,0.85,150,True,2.4332,2.3539,0.0793,0.0021,-0.0416,0.0585,0.9182,54,13,1.0252,0.8005,-0.2303
4,dev,0.90,150,True,2.0656,2.3539,-0.2883,-0.0081,-0.0367,0.0200,0.9421,54,18,0.9157,0.6907,-0.2808
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
67,stress,0.75,250,False,0.5765,0.7309,-0.1545,-0.0101,-0.0882,0.0923,0.7572,36,16,0.9135,0.4801,-0.2375
68,stress,0.80,250,False,0.8778,0.7309,0.1468,0.0088,-0.0760,0.1223,0.7620,48,27,1.0893,0.6222,-0.2132
69,stress,0.85,250,False,0.7273,0.7309,-0.0037,-0.0002,-0.0761,0.0940,0.7685,64,42,1.0037,0.5313,-0.2678
70,stress,0.90,250,False,0.5026,0.7309,-0.2283,-0.0153,-0.0742,0.0406,0.7819,93,73,0.8754,0.3960,-0.4146


In [8]:
# SAVE THE RESULTS
write_csv_file_to_path(grid_pdf, f"{output_path_str}neighbourhood_path_data.csv", "W")
write_csv_file_to_path(pd.DataFrame([neighbourhood_dict]), f"{output_path_str}neighbourhood_summary_data.csv", "W")

ℹ️ Saving In Overwrite Mode.	(💾 In Local File Storage)
⚠️ File:	'C:/Users/nico/Desktop/stock_overflow_data/store04_experiments/exp13_e1m1_robustness/step02_neighbourhood_data/neighbourhood_path_data.csv' Not Found!	(💾 In Local File Storage)
✅ File:	'C:/Users/nico/Desktop/stock_overflow_data/store04_experiments/exp13_e1m1_robustness/step02_neighbourhood_data/neighbourhood_path_data.csv' Has Been Saved!
ℹ️ Saving In Overwrite Mode.	(💾 In Local File Storage)
⚠️ File:	'C:/Users/nico/Desktop/stock_overflow_data/store04_experiments/exp13_e1m1_robustness/step02_neighbourhood_data/neighbourhood_summary_data.csv' Not Found!	(💾 In Local File Storage)
✅ File:	'C:/Users/nico/Desktop/stock_overflow_data/store04_experiments/exp13_e1m1_robustness/step02_neighbourhood_data/neighbourhood_summary_data.csv' Has Been Saved!


In [9]:
# CHECK THE TRIAL BUDGET AND THE EXPECTED COUNT BEFORE LOGGING
trial_log_pdf = read_trial_log_pdf()
check_step_trial_dict(trial_log_pdf, "step02", len(grid_pdf))
# LOG EVERY PATH AS A ROBUSTNESS TRIAL (WINDOW COLUMNS = ITS SPAN)
logged_dict_list = []
metric_col_str_list = [col for col in grid_pdf.columns if col not in ["span", "fade_ratio", "ma_length", "fresh_exit"]]
for _, row in grid_pdf.iterrows():
    logged_dict_list.append(log_trial_dict(exp_config, "robustness", {"step": "step02", "span": row["span"], "fade_ratio": float(row["fade_ratio"]), "ma_length": int(row["ma_length"]),
                                                                      "fresh_exit": bool(row["fresh_exit"])}, {col: row[col] for col in metric_col_str_list}, False,
                                           fold_row_in={"valid_start": row["first_date"], "valid_end": row["last_date"]},
                                           note_str_in="exp13 step02 parameter neighbourhood (descriptive, nothing selected)"))
print(f"Trials logged:\t{len(logged_dict_list)}\tfirst logged_ts {logged_dict_list[0]['logged_ts']}\ttest_window_touched: {sorted({row['test_window_touched'] for row in logged_dict_list})}")

Trials logged:	72	first logged_ts 2026-10-09T20:00:43	test_window_touched: [False]


In [10]:
# CALL FUNCTION TO READ THE TRIAL LOG
trial_log_pdf = read_trial_log_pdf()
# DISPLAY THE NUMBER OF TRIALS PER EXPERIMENT AND STAGE (MULTIPLE-TESTING RECORD)
get_trial_count_pdf(trial_log_pdf)

,experiment,stage,trial_count,config_count,test_touched_count,first_logged_ts,last_logged_ts
0,exp01_minute_entry,multivariate_check,1,1,0,2026-10-06T18:27:48,2026-10-06T18:27:48
1,exp01_minute_entry,signal_check,2,2,0,2026-10-05T15:17:38,2026-10-06T12:46:46
2,exp01_minute_entry,summary,1,1,0,2026-10-05T16:16:30,2026-10-05T16:16:30
3,exp01_minute_entry,validation,540,1,0,2026-10-05T15:36:53,2026-10-05T16:16:18
4,exp02_stop_reentry,exploration,14,1,0,2026-10-05T14:47:50,2026-10-05T14:47:50
5,exp02_stop_reentry,signal_check,1,1,0,2026-10-05T14:48:41,2026-10-05T14:48:41
6,exp02_stop_reentry,summary,1,1,0,2026-10-05T14:50:29,2026-10-05T14:50:29
7,exp02_stop_reentry,validation,792,1,0,2026-10-05T14:49:18,2026-10-05T14:49:48
8,exp03_ath_exit,exploration,15,1,0,2026-10-05T14:55:04,2026-10-05T14:55:04
9,exp03_ath_exit,summary,1,1,0,2026-10-05T14:56:49,2026-10-05T14:56:49
